# Import

In [18]:
#Import Modules
import pandas as pd
from google import genai
import os
import json
from dotenv import load_dotenv

# Initialize Dataset

In [19]:
#Call Datasets

train = pd.read_csv("data\MTS-Dialog-TrainingSet28SDHP%29.csv") #technically cuma perlu pake ini. validation gabakal dipake atau ngga di gabungin aja. krna this dataset was originally used for people fine tuning a model
validation = pd.read_csv("data\MTS-Dialog-Validation2029.csv")

<>:3: SyntaxWarning: "\M" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\M"? A raw string is also an option.
<>:4: SyntaxWarning: "\M" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\M"? A raw string is also an option.
<>:3: SyntaxWarning: "\M" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\M"? A raw string is also an option.
<>:4: SyntaxWarning: "\M" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\M"? A raw string is also an option.
C:\Users\justi\AppData\Local\Temp\ipykernel_20548\4035105612.py:3: SyntaxWarning: "\M" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\M"? A raw string is also an option.
  train = pd.read_csv("data\MTS-Dialog-TrainingSet28SDHP%29.csv") #technically cuma perlu pake ini. validation gabakal dipake atau ngga di gabungin aja. krna this dataset was

In [20]:
print(train.head())

   ID section_header                                       section_text  \
0   0          GENHX  Symptoms: no fever, no chills, no cough, no co...   
1   1          GENHX  Symptoms: sudden onset headache, blurry vision...   
2   2          GENHX  Symptoms: itching.\nDiagnosis: condylomas.\nHi...   
3   3    MEDICATIONS  Symptoms: N/A.\r\nDiagnosis: N/A.\r\nHistory o...   
4   4             CC  Symptoms: Burn, right arm.\r\nDiagnosis: N/A.\...   

                                            dialogue  
0  Doctor: What brings you back into the clinic t...  
1  Doctor: How're you feeling today?  \nPatient: ...  
2  Doctor: Hello, miss. What is the reason for yo...  
3  Doctor: Are you taking any over the counter me...  
4  Doctor: Hi, how are you? \nPatient: I burned m...  


In [21]:
#Only need the dialogue
dialogue = train['dialogue']

In [22]:
dialogue_1 = dialogue[0]

In [23]:
#Take first 10 dialogue and put it through an LLM (Later. Do after being able to put through 1 message to LLM)
dialogue_10 = dialogue[0:10]



# Gemini LLM Analysis

In [24]:
load_dotenv()
gemini_api = os.environ.get('GEMINI_API_KEY')
# print(gemini_api)

In [25]:
SYSTEM_PROMPT = """
You are a clinical documentation assistant trained to read doctor-patient
conversation transcripts and produce structured medical records.

Rules:
- Return ONLY a valid JSON object. No explanation, no preamble, no markdown
  code fences. The first character must be { and the last must be }.
- Use null for any field not explicitly mentioned in the transcript.
- Use [] for list fields where the topic was mentioned but nothing found.
- Never invent or infer information not clearly stated.
- For medications, capture dosage/frequency/duration exactly as spoken.
- extraction_confidence reflects transcript clarity:
  high = clear audio, complete sentences
  low  = fragmented or ambiguous dialogue

Return this exact JSON schema, following the SOAP medical format:

{
  "summary": "<2-3 sentence clinical summary of the full encounter>",
  "encounter_type": "initial_consultation | follow_up | results_review | other",

  "subjective": {
    "chief_complaint": "<patient's primary reason for the visit, in their words>",
    "symptoms": [
      "<symptom — include severity/duration if stated>"
    ],
    "symptom_onset": "<when symptoms started, e.g. '3 days ago', or null>",
    "medical_history": [
      "<relevant past condition or procedure>"
    ],
    "current_medications": [
      "<medication patient was already taking before this visit>"
    ],
    "allergies": [
      "<allergy mentioned>"
    ]
  },

  "objective": {
    "physical_exam_findings": [
      "<finding noted by doctor during examination>"
    ],
    "vital_signs": "<any vitals mentioned, e.g. 'BP 120/80', or null>"
  },

  "assessment": {
    "diagnosis": "<primary diagnosis given by the doctor, or null>",
    "differential_diagnosis": [
      "<alternative diagnosis the doctor considered>"
    ]
  },

  "plan": {
    "prescribed_medications": [
      {
        "name": "<medication name>",
        "dosage": "<e.g. 500mg, or null>",
        "frequency": "<e.g. twice daily, or null>",
        "duration": "<e.g. 7 days, or null>"
      }
    ],
    "treatment_plan": [
      "<recommendation or instruction given by the doctor>"
    ],
    "investigations_ordered": [
      "<test, lab, or imaging ordered>"
    ],
    "referrals": [
      "<specialist or department referred to>"
    ],
    "follow_up": "<follow-up instructions or timeframe, or null>"
  },

  "additional_notes": "<any clinically relevant detail not captured above, or null>",
  "extraction_confidence": "high | medium | low"
}
""".strip()

In [26]:
def build_transcript_prompt(transcript:str):
    return f"""
    Extract all clinical information from the doctor-patient conversation below.
    Follow the JSON schema exactly as specified.

    <transcript>
    {transcript.strip()}
    </transcript>
    """.strip()

In [27]:
client = genai.Client(api_key=gemini_api)
user_prompt = build_transcript_prompt(dialogue_1)
response = client.models.generate_content(
    model = 'gemini-3.1-flash-lite',
    config = genai.types.GenerateContentConfig(
        system_instruction = SYSTEM_PROMPT,
        max_output_tokens = 1024
    ), 
    contents = user_prompt
)
print(response)
print("------")
print(response.text)

sdk_http_response=HttpResponse(
  headers=<dict len=12>
) candidates=[Candidate(
  content=Content(
    parts=[
      Part(
        text="""{
  "summary": "A 76-year-old female presents for a medication refill for hypertension. The patient reports no new concerns, symptoms, or acute illness during the review of systems.",
  "encounter_type": "follow_up",
  "subjective": {
    "chief_complaint": "refill of my blood pressure medicine",
    "symptoms": [],
    "symptom_onset": null,
    "medical_history": [
      "hypertension",
      "osteoarthritis",
      "osteoporosis",
      "hypothyroidism",
      "allergic rhinitis",
      "kidney stones"
    ],
    "current_medications": [],
    "allergies": []
  },
  "objective": {
    "physical_exam_findings": [],
    "vital_signs": null
  },
  "assessment": {
    "diagnosis": "hypertension",
    "differential_diagnosis": []
  },
  "plan": {
    "prescribed_medications": [],
    "treatment_plan": [
      "Process refill for blood pressure medica

In [28]:
print(response.text)

{
  "summary": "A 76-year-old female presents for a medication refill for hypertension. The patient reports no new concerns, symptoms, or acute illness during the review of systems.",
  "encounter_type": "follow_up",
  "subjective": {
    "chief_complaint": "refill of my blood pressure medicine",
    "symptoms": [],
    "symptom_onset": null,
    "medical_history": [
      "hypertension",
      "osteoarthritis",
      "osteoporosis",
      "hypothyroidism",
      "allergic rhinitis",
      "kidney stones"
    ],
    "current_medications": [],
    "allergies": []
  },
  "objective": {
    "physical_exam_findings": [],
    "vital_signs": null
  },
  "assessment": {
    "diagnosis": "hypertension",
    "differential_diagnosis": []
  },
  "plan": {
    "prescribed_medications": [],
    "treatment_plan": [
      "Process refill for blood pressure medication"
    ],
    "investigations_ordered": [],
    "referrals": [],
    "follow_up": null
  },
  "additional_notes": "Patient is 76 years ol

In [29]:
print(response.usage_metadata)

cache_tokens_details=None cached_content_token_count=None candidates_token_count=314 candidates_tokens_details=None prompt_token_count=847 prompt_tokens_details=[ModalityTokenCount(
  modality=<MediaModality.TEXT: 'TEXT'>,
  token_count=847
)] thoughts_token_count=None tool_use_prompt_token_count=None tool_use_prompt_tokens_details=None total_token_count=1161 traffic_type=None


In [30]:
print(dialogue_1)

Doctor: What brings you back into the clinic today, miss? 
Patient: I came in for a refill of my blood pressure medicine. 
Doctor: It looks like Doctor Kumar followed up with you last time regarding your hypertension, osteoarthritis, osteoporosis, hypothyroidism, allergic rhinitis and kidney stones.  Have you noticed any changes or do you have any concerns regarding these issues?  
Patient: No. 
Doctor: Have you had any fever or chills, cough, congestion, nausea, vomiting, chest pain, chest pressure?
Patient: No.  
Doctor: Great. Also, for our records, how old are you and what race do you identify yourself as?
Patient: I am seventy six years old and identify as a white female.


# Convert Response to CSV

In [31]:
print(type(response.text))

<class 'str'>


In [32]:
response_dict = json.loads(response.text)
print(response_dict)
print(type(response_dict))

{'summary': 'A 76-year-old female presents for a medication refill for hypertension. The patient reports no new concerns, symptoms, or acute illness during the review of systems.', 'encounter_type': 'follow_up', 'subjective': {'chief_complaint': 'refill of my blood pressure medicine', 'symptoms': [], 'symptom_onset': None, 'medical_history': ['hypertension', 'osteoarthritis', 'osteoporosis', 'hypothyroidism', 'allergic rhinitis', 'kidney stones'], 'current_medications': [], 'allergies': []}, 'objective': {'physical_exam_findings': [], 'vital_signs': None}, 'assessment': {'diagnosis': 'hypertension', 'differential_diagnosis': []}, 'plan': {'prescribed_medications': [], 'treatment_plan': ['Process refill for blood pressure medication'], 'investigations_ordered': [], 'referrals': [], 'follow_up': None}, 'additional_notes': 'Patient is 76 years old, white female.', 'extraction_confidence': 'high'}
<class 'dict'>


In [33]:
print(response_dict.keys())
print(response_dict.values())

dict_keys(['summary', 'encounter_type', 'subjective', 'objective', 'assessment', 'plan', 'additional_notes', 'extraction_confidence'])
dict_values(['A 76-year-old female presents for a medication refill for hypertension. The patient reports no new concerns, symptoms, or acute illness during the review of systems.', 'follow_up', {'chief_complaint': 'refill of my blood pressure medicine', 'symptoms': [], 'symptom_onset': None, 'medical_history': ['hypertension', 'osteoarthritis', 'osteoporosis', 'hypothyroidism', 'allergic rhinitis', 'kidney stones'], 'current_medications': [], 'allergies': []}, {'physical_exam_findings': [], 'vital_signs': None}, {'diagnosis': 'hypertension', 'differential_diagnosis': []}, {'prescribed_medications': [], 'treatment_plan': ['Process refill for blood pressure medication'], 'investigations_ordered': [], 'referrals': [], 'follow_up': None}, 'Patient is 76 years old, white female.', 'high'])


In [34]:
response_norm = pd.json_normalize(response_dict)
response_df = pd.DataFrame.from_dict(response_norm)
print(response_df)
print(type(response_df))

                                             summary encounter_type  \
0  A 76-year-old female presents for a medication...      follow_up   

                         additional_notes extraction_confidence  \
0  Patient is 76 years old, white female.                  high   

             subjective.chief_complaint subjective.symptoms  \
0  refill of my blood pressure medicine                  []   

  subjective.symptom_onset                         subjective.medical_history  \
0                     None  [hypertension, osteoarthritis, osteoporosis, h...   

  subjective.current_medications subjective.allergies  \
0                             []                   []   

  objective.physical_exam_findings objective.vital_signs assessment.diagnosis  \
0                               []                  None         hypertension   

  assessment.differential_diagnosis plan.prescribed_medications  \
0                                []                          []   

                   

In [35]:
response_df.to_csv('medical_report.csv', index=False)